# DRAFT: Pelatihan End-to-End, Ekspor GGUF, dan Evaluasi di Google Colab

Notebook ini menjalankan seluruh rangkaian pipeline **DRAFT** dalam sekali jalan (`Runtime -> Run all`) serta mencadangkan checkpoint, indeks retrieval, model GGUF, dan laporan evaluasi ke Google Drive setiap selesai satu tahap.

### Persiapan Sebelum Menjalankan
1. Pilih salah satu sumber kode proyek pada Cell konfigurasi di bawah:
   - Isi `GITHUB_REPO_URL` untuk langsung `git clone` dari repositori GitHub, **atau**
   - Unggah folder proyek (`configs/`, `prompts/`, `scripts/`, `src/`, `data/org_docs/`, `pyproject.toml`) ke `MyDrive/draft-rag/`, **atau** unggah `team-project-src.zip` ke dalam `MyDrive/draft-rag/`.
2. Aktifkan runtime GPU: **Runtime -> Change runtime type -> Hardware accelerator: GPU** (A100 40GB disarankan; L4 dan T4 juga didukung otomatis).
3. Jalankan **Runtime -> Run all**.

### Keluaran yang Disimpan ke Google Drive (`MyDrive/draft-rag/`)
- `data/processed/`: split kanonik `train.parquet`, `val.parquet`, `test.parquet`, dan dataset SFT JSONL (`sft/`)
- `artifacts/classifier_baseline/` & `artifacts/classifier/`: Stage 1 TF-IDF baseline dan fine-tuned `microsoft/deberta-v3-small`
- `artifacts/retriever/`: Stage 2 fine-tuned `BAAI/bge-small-en-v1.5` dan indeks ganda FAISS + BM25
- `artifacts/llm/`: Stage 3 adapter LoRA `Qwen2.5-3B-Instruct`, `Modelfile`, dan `draft-qwen25-3b-q8_0.gguf` untuk Ollama lokal
- `reports/`: ringkasan data, metrik klasifikasi, metrik retrieval, metrik training LLM, dan `ablation_table.json`

## 1. Konfigurasi Eksekusi
Sesuaikan path Google Drive, URL GitHub (opsional), dan tahap yang ingin dijalankan. Saat `SKIP_IF_DONE = True`, checkpoint yang sudah tersimpan di Drive akan langsung dipakai ulang tanpa melatih ulang.

In [1]:
# Konfigurasi yang dapat disesuaikan
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/draft-rag"
GITHUB_REPO_URL = "https://github.com/karolyangqian/team-project-nlp.git"  # Opsional: URL repo GitHub untuk git clone langsung
GITHUB_BRANCH = "refactor/cleanup"
DRIVE_ZIP_PATH = ""  # Opsional: path eksplisit arsip .zip di Google Drive
WORK_DIR = "/content/draft-rag"

SKIP_IF_DONE = True  # Pakai ulang checkpoint yang sudah ada di Drive

RUN_STAGE1 = True
RUN_STAGE2 = True
RUN_STAGE3 = True
RUN_GGUF_EXPORT = True
RUN_ABLATION = True

EVAL_CLASSIFIER_BACKEND = "baseline"  # Pilihan: "baseline", "transformer", atau "auto"
SFT_TICKET_SAMPLES = 3000
ABLATION_SAMPLES = 50

## 2. Mount Google Drive & Deteksi Profil GPU
Ukuran batch dan mode kuantisasi diatur otomatis mengikuti kapasitas VRAM GPU (A100/H100, L4, atau T4).

In [2]:
import subprocess
import torch
from google.colab import drive

drive.mount("/content/drive")

subprocess.run(["nvidia-smi"], check=True)
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU CUDA tidak terdeteksi. Aktifkan GPU melalui Runtime -> Change runtime type."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
try:
    bf16_ok = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    bf16_ok = torch.cuda.is_bf16_supported()

if vram_gb >= 32 and bf16_ok:
    GPU_PROFILE = "A100/H100 BF16"
    CLS_BATCH = 64
    RET_BATCH = 128
    LLM_BATCH = 8
    LLM_GRAD_ACCUM = 2
    LLM_NO_4BIT = True
elif vram_gb >= 20 and bf16_ok:
    GPU_PROFILE = "L4 BF16 QLoRA"
    CLS_BATCH = 32
    RET_BATCH = 64
    LLM_BATCH = 4
    LLM_GRAD_ACCUM = 4
    LLM_NO_4BIT = False
else:
    GPU_PROFILE = "T4 QLoRA"
    CLS_BATCH = 32
    RET_BATCH = 64
    LLM_BATCH = 2
    LLM_GRAD_ACCUM = 8
    LLM_NO_4BIT = False

print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM, bf16={bf16_ok}) -> Profil: {GPU_PROFILE}")
print(
    f"Batch otomatis: CLS={CLS_BATCH}, RET={RET_BATCH}, "
    f"LLM={LLM_BATCH}x{LLM_GRAD_ACCUM}, no_4bit={LLM_NO_4BIT}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4 (14.6 GB VRAM, bf16=False) -> Profil: T4 QLoRA
Batch otomatis: CLS=32, RET=64, LLM=2x8, no_4bit=False


## 3. Sinkronisasi Kode Proyek & Instalasi Dependensi
Menyalin kode sumber ke disk lokal Colab agar I/O lebih cepat, lalu memasang seluruh paket Python yang dibutuhkan.

In [3]:
import importlib
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

drive_dir = Path(DRIVE_PROJECT_DIR)
work_dir = Path(WORK_DIR)
drive_dir.mkdir(parents=True, exist_ok=True)

if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True, exist_ok=True)

zip_candidate = Path(DRIVE_ZIP_PATH) if DRIVE_ZIP_PATH else (drive_dir / "team-project-src.zip")
if GITHUB_REPO_URL.strip():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            GITHUB_BRANCH,
            GITHUB_REPO_URL.strip(),
            str(work_dir),
        ],
        check=True,
    )
    for resumable in ("data/processed", "artifacts", "reports"):
        src_res = drive_dir / resumable
        if src_res.exists():
            shutil.copytree(src_res, work_dir / resumable, dirs_exist_ok=True)
elif (drive_dir / "src").exists() and not DRIVE_ZIP_PATH:
    for item in (
        "configs", "prompts", "scripts", "src", "data", "pyproject.toml", "artifacts", "reports"
    ):
        src_path = drive_dir / item
        if src_path.exists():
            dst_path = work_dir / item
            if src_path.is_dir():
                shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            else:
                shutil.copy2(src_path, dst_path)
elif zip_candidate.exists():
    with zipfile.ZipFile(zip_candidate, "r") as zf:
        zf.extractall(work_dir)
    if not (work_dir / "src").exists():
        nested = [p for p in work_dir.iterdir() if p.is_dir() and (p / "src").exists()]
        if len(nested) == 1:
            for child in nested[0].iterdir():
                shutil.move(str(child), str(work_dir / child.name))
            nested[0].rmdir()
    for resumable in ("data/processed", "artifacts", "reports"):
        src_res = drive_dir / resumable
        if src_res.exists():
            shutil.copytree(src_res, work_dir / resumable, dirs_exist_ok=True)
else:
    raise FileNotFoundError(
        f"Sumber proyek tidak ditemukan di {drive_dir} maupun {zip_candidate}, "
        "dan GITHUB_REPO_URL kosong."
    )

os.chdir(work_dir)
os.environ["PYTHONUNBUFFERED"] = "1"
print("Direktori kerja:", Path.cwd())


def run_step(cmd: list[str]) -> None:
    """Jalankan perintah subprocess dengan output streaming real-time."""
    print(">>>", " ".join(cmd))
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Perintah gagal (exit code {code}): {' '.join(cmd)}")


def sync_to_drive(rel_path: str) -> None:
    """Salin file atau folder hasil tahap secara inkremental ke Google Drive."""
    src = work_dir / rel_path
    dst = drive_dir / rel_path
    if not src.exists():
        print(f"Lewati sinkronisasi Drive (tidak ada): {rel_path}")
        return
    if src.is_dir():
        shutil.copytree(
            src,
            dst,
            ignore=shutil.ignore_patterns("checkpoints", "__pycache__", "*.tmp"),
            dirs_exist_ok=True,
        )
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f"Tersimpan ke Drive: {dst}")


pkgs = [
    "trl", "peft", "bitsandbytes", "accelerate", "datasets", "transformers",
    "sentence-transformers", "safetensors", "sentencepiece", "protobuf",
    "pydantic", "pydantic-settings", "pyyaml", "jinja2", "typer", "httpx",
    "rank-bm25", "faiss-cpu", "rouge-score", "bert-score", "evaluate", "gguf",
]
run_step([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs])
run_step([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."])
src_path_str = str((work_dir / "src").resolve())
if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)
importlib.invalidate_caches()
print("Paket `draft` siap diimpor dari:", src_path_str)

Direktori kerja: /content/draft-rag
>>> /usr/bin/python3 -m pip install -q --upgrade trl peft bitsandbytes accelerate datasets transformers sentence-transformers safetensors sentencepiece protobuf pydantic pydantic-settings pyyaml jinja2 typer httpx rank-bm25 faiss-cpu rouge-score bert-score evaluate gguf
>>> /usr/bin/python3 -m pip install -q --no-deps -e .
Paket `draft` siap diimpor dari: /content/draft-rag/src


## 4. Persiapan Data & Baseline TF-IDF
Mengunduh dataset tiket dari Hugging Face, memfilter bahasa Inggris, menghapus duplikat, membuat split berstrata 80/10/10, serta melatih baseline TF-IDF + Logistic Regression.

In [4]:
import sys
from pathlib import Path

splits_ready = all(
    Path(f"data/processed/{name}.parquet").exists() for name in ("train", "val", "test")
)
if not splits_ready:
    run_step([sys.executable, "-m", "draft.cli", "prepare-data"])
    sync_to_drive("data/processed/train.parquet")
    sync_to_drive("data/processed/val.parquet")
    sync_to_drive("data/processed/test.parquet")
    sync_to_drive("reports/data_summary.json")
else:
    print("Split kanonik sudah tersedia.")

baseline_done = Path("artifacts/classifier_baseline/vectorizer.joblib").exists()
if not (SKIP_IF_DONE and baseline_done):
    run_step([sys.executable, "-m", "draft.cli", "train-baseline"])
    sync_to_drive("artifacts/classifier_baseline")
    sync_to_drive("reports/classifier_baseline_metrics.json")
else:
    print("Melewati baseline TF-IDF (sudah tersedia).")

Split kanonik sudah tersedia.
>>> /usr/bin/python3 -m draft.cli train-baseline
TF-IDF baseline training completed:
  [val] intent(acc=0.5485, macro_f1=0.5696, weighted_f1=0.5498) | urgency(acc=0.6325, macro_f1=0.6231, weighted_f1=0.6336)
  [test] intent(acc=0.5304, macro_f1=0.5458, weighted_f1=0.5316) | urgency(acc=0.6034, macro_f1=0.5906, weighted_f1=0.6058)
  [model] artifacts/classifier_baseline
  [report] reports/classifier_baseline_metrics.json
Tersimpan ke Drive: /content/drive/MyDrive/draft-rag/artifacts/classifier_baseline
Tersimpan ke Drive: /content/drive/MyDrive/draft-rag/reports/classifier_baseline_metrics.json


## 5. Stage 1 — Pelatihan Multi-Task Classifier (`DeBERTa-v3-small`)
Melatih klasifikator multi-task (10 kelas intent + 3 level urgensi) dengan penjagaan FP32 otomatis pada GPU T4 agar perhatian terdisentang DeBERTa tidak mengalami overflow.

In [5]:
import json
import sys
from pathlib import Path
import pandas as pd

cls_ckpt = Path("artifacts/classifier/model.safetensors")
if RUN_STAGE1:
    if SKIP_IF_DONE and cls_ckpt.exists():
        print("Melewati training Stage 1 transformer (checkpoint sudah tersedia).")
    else:
        run_step(
            [
                sys.executable,
                "scripts/colab/train_classifier.py",
                "--config",
                "configs/default.yaml",
                "--output-dir",
                "artifacts/classifier",
                "--batch-size",
                str(CLS_BATCH),
            ]
        )
        sync_to_drive("artifacts/classifier")
        sync_to_drive("reports/classifier_transformer_metrics.json")

rows = []
for name, path in (
    ("TF-IDF Baseline", Path("reports/classifier_baseline_metrics.json")),
    ("DeBERTa-v3-small", Path("reports/classifier_transformer_metrics.json")),
):
    if path.exists():
        data = json.loads(path.read_text(encoding="utf-8"))
        for split in ("val", "test"):
            m = data[split]
            rows.append(
                {
                    "Model": name,
                    "Split": split,
                    "Intent Acc": round(m["intent"]["accuracy"], 4),
                    "Intent Macro-F1": round(m["intent"]["macro_f1"], 4),
                    "Urgency Acc": round(m["urgency"]["accuracy"], 4),
                    "Urgency Macro-F1": round(m["urgency"]["macro_f1"], 4),
                    "Combined Macro-F1": round(m["combined_macro_f1"], 4),
                }
            )
if rows:
    display(pd.DataFrame(rows))

Melewati training Stage 1 transformer (checkpoint sudah tersedia).


,Model,Split,Intent Acc,Intent Macro-F1,Urgency Acc,Urgency Macro-F1,Combined Macro-F1
0,TF-IDF Baseline,val,0.5485,0.5696,0.6325,0.6231,0.5963
1,TF-IDF Baseline,test,0.5304,0.5458,0.6034,0.5906,0.5682
2,DeBERTa-v3-small,val,0.3996,0.4031,0.5430,0.5369,0.4700
3,DeBERTa-v3-small,test,0.3979,0.3950,0.5274,0.5188,0.4569


## 6. Stage 2 — Fine-Tuning Bi-Encoder (`BAAI/bge-small-en-v1.5`) & Indeks Dual-Source
Menjalankan contrastive fine-tuning dengan `MultipleNegativesRankingLoss`, membangun ulang indeks FAISS + BM25 (42 dokumen kebijakan + 18.959 tiket latih), lalu mengevaluasi ablasi retrieval pada split `test`.

In [6]:
import sys
from pathlib import Path

STAGE2_RAN = False
enc_dir = Path("artifacts/retriever/encoder")
idx_meta = Path("artifacts/retriever/index/index_meta.json")

if RUN_STAGE2:
    if SKIP_IF_DONE and (enc_dir / "config.json").exists() and idx_meta.exists():
        print("Melewati training Stage 2 bi-encoder (encoder dan indeks sudah tersedia).")
    else:
        run_step(
            [
                sys.executable,
                "scripts/colab/train_retriever.py",
                "--config",
                "configs/default.yaml",
                "--output-dir",
                str(enc_dir),
                "--batch-size",
                str(RET_BATCH),
                "--epochs",
                "2",
                "--rebuild-index",
            ]
        )
        STAGE2_RAN = True
        sync_to_drive("artifacts/retriever")
        sync_to_drive("reports/retriever_training_metrics.json")

    if STAGE2_RAN or not Path("reports/retrieval_metrics.json").exists():
        run_step(
            [
                sys.executable,
                "-m",
                "draft.cli",
                "eval-retrieval",
                "--split",
                "test",
                "--max-queries",
                "300",
                "--rerank",
            ]
        )
        sync_to_drive("reports/retrieval_metrics.json")

Melewati training Stage 2 bi-encoder (encoder dan indeks sudah tersedia).


## 7. Stage 3 — Pembangunan Dataset SFT & Fine-Tuning LoRA (`Qwen2.5-3B-Instruct`)
Membentuk dataset RAG SFT bebas kebocoran data (`exclude_email_ids`) dari tiket latih dan contoh dokumen kebijakan, lalu melatih adapter LoRA serta membuat `Modelfile` Ollama.

In [7]:
import sys
from pathlib import Path

sft_train = Path("data/processed/sft/sft_train.jsonl")
need_sft = (not SKIP_IF_DONE) or STAGE2_RAN or (not sft_train.exists())

if RUN_STAGE3 and need_sft:
    run_step(
        [
            sys.executable,
            "-m",
            "draft.cli",
            "build-sft-data",
            "--num-ticket-samples",
            str(SFT_TICKET_SAMPLES),
            "--no-rerank",
        ]
    )
    sync_to_drive("data/processed/sft")
else:
    print("Menggunakan dataset SFT yang sudah tersedia.")

Menggunakan dataset SFT yang sudah tersedia.


In [8]:
import sys
from pathlib import Path

adapter_cfg = Path("artifacts/llm/lora_adapter/adapter_config.json")
if RUN_STAGE3:
    if SKIP_IF_DONE and adapter_cfg.exists():
        print("Melewati fine-tuning LLM Stage 3 (adapter LoRA sudah tersedia).")
    else:
        cmd = [
            sys.executable,
            "scripts/colab/train_llm.py",
            "--config",
            "configs/default.yaml",
            "--sft-dir",
            "data/processed/sft",
            "--output-dir",
            "artifacts/llm/lora_adapter",
            "--epochs",
            "2",
            "--batch-size",
            str(LLM_BATCH),
            "--grad-accum",
            str(LLM_GRAD_ACCUM),
            "--max-seq-length",
            "1536",
            "--merge-fp16",
        ]
        if LLM_NO_4BIT:
            cmd.append("--no-4bit")
        run_step(cmd)
        sync_to_drive("artifacts/llm/lora_adapter")
        sync_to_drive("artifacts/llm/Modelfile")
        sync_to_drive("reports/llm_training_metrics.json")

Melewati fine-tuning LLM Stage 3 (adapter LoRA sudah tersedia).


## 8. Ekspor Model ke GGUF (`Q8_0`) untuk Ollama Lokal
Menggabungkan adapter LoRA ke bobot dasar FP16 lalu mengonversinya menjadi satu berkas `draft-qwen25-3b-q8_0.gguf` melalui skrip Python `llama.cpp`.

In [9]:
import gc
import os
import shutil
import sys
from pathlib import Path
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

work_root = Path(WORK_DIR) if "WORK_DIR" in globals() else Path("/content/draft-rag")
if Path.cwd() != work_root and work_root.exists():
    os.chdir(work_root)
src_dir = str((Path.cwd() / "src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from draft.config import load_config

cfg = load_config("configs/default.yaml")
gguf_path = Path(f"artifacts/llm/{cfg.generation.ollama_model}-q8_0.gguf")
merged_dir = Path("artifacts/llm/merged_fp16")
adapter_dir = Path("artifacts/llm/lora_adapter")

if RUN_GGUF_EXPORT:
    if SKIP_IF_DONE and gguf_path.exists():
        print("Melewati ekspor GGUF (berkas sudah tersedia).")
    else:
        llama_dir = Path("/content/llama.cpp")
        if not llama_dir.exists():
            run_step(
                ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(llama_dir)]
            )

        if not (merged_dir / "config.json").exists():
            if not adapter_dir.exists():
                raise FileNotFoundError(f"Adapter LoRA tidak ditemukan di {adapter_dir}")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            base = AutoModelForCausalLM.from_pretrained(
                cfg.generation.base_model,
                torch_dtype=torch.float16,
                device_map="cpu",
            )
            merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()
            merged_dir.mkdir(parents=True, exist_ok=True)
            merged.save_pretrained(str(merged_dir), safe_serialization=True)
            AutoTokenizer.from_pretrained(str(adapter_dir)).save_pretrained(str(merged_dir))
            del base, merged
            gc.collect()

        run_step(
            [
                sys.executable,
                str(llama_dir / "convert_hf_to_gguf.py"),
                str(merged_dir),
                "--outfile",
                str(gguf_path),
                "--outtype",
                "q8_0",
            ]
        )
        sync_to_drive(str(gguf_path))
        sync_to_drive("artifacts/llm/Modelfile")

    if merged_dir.exists():
        shutil.rmtree(merged_dir, ignore_errors=True)
        print("Direktori sementara merged_fp16 dihapus untuk menghemat ruang disk.")

Melewati ekspor GGUF (berkas sudah tersedia).


## 9. Evaluasi Ablasi 4-Mode & Ringkasan Hasil
Membandingkan `base_no_rag`, `base_with_rag_0shot`, `base_with_rag_2shot`, dan `finetuned_with_rag` pada split `test` serta 12 skenario kebijakan tak terlihat.

In [10]:
import sys
from pathlib import Path

ablation_report = Path("reports/ablation_table.json")
if RUN_ABLATION:
    if SKIP_IF_DONE and ablation_report.exists():
        print("Melewati ablasi 4-mode (reports/ablation_table.json sudah tersedia).")
    else:
        cmd = [
            sys.executable,
            "scripts/colab/eval_ablation.py",
            "--config",
            "configs/default.yaml",
            "--split",
            "test",
            "--max-samples",
            str(ABLATION_SAMPLES),
            "--classifier-backend",
            EVAL_CLASSIFIER_BACKEND,
            "--rerank",
            "--bertscore",
        ]
        if LLM_NO_4BIT:
            cmd.append("--no-4bit")
        run_step(cmd)
        sync_to_drive("reports/ablation_table.json")

>>> /usr/bin/python3 scripts/colab/eval_ablation.py --config configs/default.yaml --split test --max-samples 50 --classifier-backend baseline --rerank --bertscore

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 855.38it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13086.56it/s]

Fetching 2 files: 100%|██████████| 2/2 [01:01<00:00, 30.70s/it]

Loading weights: 100%|██████████| 434/434 [00:20<00:00, 21.60it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9647.18it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

In [11]:
import json
from pathlib import Path
import pandas as pd

ret_report = Path("reports/retrieval_metrics.json")
if ret_report.exists():
    ret_data = json.loads(ret_report.read_text(encoding="utf-8"))
    print("Hasil Ablasi Retrieval Stage 2 (Test):")
    org_rows = [
        {
            "Mode": mode,
            "Category Hit@1": round(m["category_hit@1"], 4),
            "Category Hit@3": round(m["category_hit@3"], 4),
            "Category MRR@3": round(m["category_mrr@3"], 4),
            "Unique Docs": m["unique_docs_retrieved"],
        }
        for mode, m in ret_data["org_docs"].items()
    ]
    display(pd.DataFrame(org_rows))

abl_report = Path("reports/ablation_table.json")
if abl_report.exists():
    abl = json.loads(abl_report.read_text(encoding="utf-8"))
    print(f"Hasil Ablasi Generasi End-to-End (split={abl['split']}, n={abl['max_samples']}):")
    abl_rows = []
    for mode, data in abl["modes"].items():
        t = data["tickets"]
        p = data["policy_suite"]
        abl_rows.append(
            {
                "Mode": mode,
                "ROUGE-1": round(t["rouge"]["rouge1_f1"], 4),
                "ROUGE-L": round(t["rouge"]["rougeL_f1"], 4),
                "BERTScore F1": round(t.get("bertscore", {}).get("f1", 0.0), 4),
                "Citation Rate": round(t["policy_adherence"]["citation_rate"], 4),
                "Leak Rate": round(
                    t["policy_adherence"].get("placeholder_leakage_rate", 0.0), 4
                ),
                "Policy Doc Hit": round(p["doc_retrieval_hit_rate"], 4),
                "Policy Fact Recall": round(p["policy_fact_recall"], 4),
                "Mean Latency (ms)": round(t["latency_ms"]["total_ms"]["mean"], 1),
            }
        )
    display(pd.DataFrame(abl_rows))

drive_root = Path(DRIVE_PROJECT_DIR)
print("\nRingkasan Artefak di Google Drive:")
for folder in ("artifacts", "reports", "data/processed"):
    target = drive_root / folder
    if target.exists():
        files = sorted(p for p in target.rglob("*") if p.is_file())
        total_mb = sum(p.stat().st_size for p in files) / (1024 ** 2)
        print(f"  {target} -> {len(files)} file ({total_mb:.1f} MB)")

Hasil Ablasi Retrieval Stage 2 (Test):


,Mode,Category Hit@1,Category Hit@3,Category MRR@3,Unique Docs
0,bm25_only,0.4633,0.7433,0.5833,41
1,dense_only,0.5233,0.7900,0.6411,42
2,hybrid,0.5600,0.7900,0.6611,42
3,hybrid_boosted,0.9600,0.9933,0.9739,42
4,hybrid_boosted_reranked,0.5833,0.7933,0.6728,42


Hasil Ablasi Generasi End-to-End (split=test, n=50):


,Mode,ROUGE-1,ROUGE-L,BERTScore F1,Citation Rate,Leak Rate,Policy Doc Hit,Policy Fact Recall,Mean Latency (ms)
0,base_no_rag,0.2684,0.1521,0.7717,0.00,0.00,0.0,0.2083,23354.4
1,base_with_rag_0shot,0.3026,0.1976,0.7881,0.40,0.42,1.0,0.8333,25914.8
2,base_with_rag_2shot,0.3864,0.2720,0.8146,0.26,0.46,1.0,0.8889,19682.4
3,finetuned_with_rag,0.5931,0.4887,0.8853,0.00,0.52,1.0,0.5417,15157.6



Ringkasan Artefak di Google Drive:
  /content/drive/MyDrive/draft-rag/artifacts -> 33 file (3992.7 MB)
  /content/drive/MyDrive/draft-rag/reports -> 6 file (0.1 MB)
  /content/drive/MyDrive/draft-rag/data/processed -> 6 file (29.8 MB)
